# GBM Rb-E2F ODE Model**Run order:**1. `GBM_RbE2F_preproc.R` → generates `GBM_patient_RbE2Fgenes.csv`2. This notebook → generates E2F/phospho-Rb activity outputs for TCGA-GBM patients3. `GBM_RbE2F_TCGA_discovery.R` → Cox/KM scan to find the best candidate feature (discovery only)**Model:** Yao et al. 2008, "A bistable Rb-E2F switch underlies the restriction point"(Nat Cell Biol), BioModels BIOMD0000000318. 7 species, 17 reactions — equations andparameters below were extracted directly from the SBML (not re-derived from memory)to avoid transcription errors.**Outputs:**- `GBM_RbE2F_EF_patients.csv` — active E2F per patient per serum-stimulus (S) level- `GBM_RbE2F_RP_patients.csv` — phosphorylated Rb per patient per S level**S grid:** 10 log-spaced levels from 0.01 to 5 (chosen after testing: the switchthreshold shifts substantially with patient gene expression — e.g. an RB1-highpatient may not switch on until S~2, while an RB1-low/MYC-high patient can alreadybe switched on at S=0.01 — so a log-spaced grid resolves the transition acrossthat range better than a linear one would).This produces 20 candidate features per patient (10 S levels × 2 readouts: E2F, phospho-Rb).

In [ ]:
import numpy as np
import pandas as pd
import scipy.integrate
import pylab as plt
import os

directory = os.getcwd()
print('Working directory:', directory)

In [ ]:
# Fixed kinetic parameters (verified directly from BIOMD0000000318 SBML — species,
# reactions, and kinetic-law parameters extracted with libsbml, not transcribed from
# the paper text, to avoid errors)

KS      = 0.5
kM      = 1.0
kkCDS   = 0.45
KEF     = 0.15
KMC     = 0.15
kkEF    = 0.4
kkb     = 0.003
kkCE    = 0.35
kkCD    = 0.03
kkRB    = 0.18
KD      = 0.92
KE      = 0.92
kkRBPP  = 18.0
kkRE    = 180.0
kkRBP   = 18.0
kkRBP2  = 18.0
kkRBUP  = 3.6
Kp      = 0.01
dMC     = 0.7
dEF     = 0.25
dCE     = 1.5
dCD     = 1.5
dRB     = 0.06
dRP     = 0.06
dRE     = 0.03

# Reference initial conditions (from SBML: all species start at 0 except the
# Rb-E2F complex, consistent with a quiescent starting state where Rb sequesters E2F)
# Species order: [MC, EF, CD, CE, RB, RE, RP]
x0_ref = [0.0, 0.0, 0.0, 0.0, 0.0, 0.55, 0.0]

print('Baseline parameters loaded from SBML.')

In [ ]:
# Patient-specific parameter function
#
# Mirrors the approach used in GBM_p53_model.ipynb: patient RNA-seq (relative
# expression) scales the SYNTHESIS-RATE constant for the gene that specific
# species corresponds to, while Michaelis constants, phosphorylation/binding
# rates, and degradation rates stay fixed at the published reference values.
# Only 5 of the 7 species map to a single specific human gene (the model lumps
# CyclinD/CDK4,6 and CyclinE/CDK2 into single catalytic species "CD"/"CE"):
#   MYC   -> kM      (Myc synthesis)
#   CCND1 -> kkCDS, kkCD  (both CycD-producing routes)
#   CCNE1 -> kkCE    (CycE synthesis)
#   RB1   -> kkRB    (Rb synthesis)
#   E2F1  -> kkEF, kkb    (both E2F-producing routes)

def get_patient_rates(sample_row):
    kM_     = kM     * sample_row['MYC']
    kkCDS_  = kkCDS  * sample_row['CCND1']
    kkCD_   = kkCD   * sample_row['CCND1']
    kkCE_   = kkCE   * sample_row['CCNE1']
    kkRB_   = kkRB   * sample_row['RB1']
    kkEF_   = kkEF   * sample_row['E2F1']
    kkb_    = kkb    * sample_row['E2F1']
    return kM_, kkCDS_, kkCD_, kkCE_, kkRB_, kkEF_, kkb_

print('get_patient_rates() defined.')

In [ ]:
# ODE system (derived directly from the SBML reaction list + kinetic laws;
# verified numerically to reproduce the expected bistable switch behaviour and
# correct-direction sensitivity to RB1 dosage before use)

def f(x, t, S, rates):
    MC, EF, CD, CE, RB, RE, RP = x
    kM_, kkCDS_, kkCD_, kkCE_, kkRB_, kkEF_, kkb_ = rates

    dMCdt = kM_*S/(KS+S) - dMC*MC
    dCDdt = kkCDS_*S/(KS+S) + kkCD_*MC/(KMC+MC) - dCD*CD
    dEFdt = (kkEF_*MC*EF/((KMC+MC)*(KEF+EF)) + kkb_*MC/(KMC+MC)
             + kkRBPP*CD*RE/(KD+RE) + kkRBPP*CE*RE/(KE+RE)
             - kkRE*RB*EF - dEF*EF)
    dCEdt = kkCE_*EF/(KEF+EF) - dCE*CE
    dRBdt = (kkRB_ - kkRE*RB*EF - kkRBP*CD*RB/(KD+RB) - kkRBP2*CE*RB/(KE+RB)
             + kkRBUP*RP/(Kp+RP) - dRB*RB)
    dREdt = kkRE*RB*EF - kkRBPP*CD*RE/(KD+RE) - kkRBPP*CE*RE/(KE+RE) - dRE*RE
    dRPdt = (kkRBPP*CD*RE/(KD+RE) + kkRBPP*CE*RE/(KE+RE)
             + kkRBP*CD*RB/(KD+RB) + kkRBP2*CE*RB/(KE+RB)
             - kkRBUP*RP/(Kp+RP) - dRP*RP)

    return [dMCdt, dEFdt, dCDdt, dCEdt, dRBdt, dREdt, dRPdt]

print('ODE system f() defined.')

In [ ]:
# Simulation function: run ODE for a set of samples
#
# S grid is log-spaced (not linear like the p53 model's DDR grid) because testing
# showed the switch threshold moves substantially with patient gene expression —
# a linear grid over a wide enough range to catch all patients would under-sample
# the transition region for most of them.

Svec           = np.logspace(np.log10(0.01), np.log10(5), num=10)
Tfinish        = 150
numberOfPoints = 50
tspan          = np.linspace(0, Tfinish, num=numberOfPoints)

def run_ode_for_cohort(params_df, label='cohort'):
    col_names = ['S_' + '{:.3f}'.format(s) for s in Svec]
    EF_df = pd.DataFrame(columns=col_names)
    RP_df = pd.DataFrame(columns=col_names)

    for i in range(len(params_df)):
        row = params_df.iloc[i]
        rates = get_patient_rates(row)
        EF_row, RP_row = [], []
        for Sval in Svec:
            sol = scipy.integrate.odeint(f, x0_ref, tspan, args=(Sval, rates), mxstep=5000)
            EF_row.append(sol[-1, 1])  # EF (active E2F) at final time
            RP_row.append(sol[-1, 6])  # RP (phosphorylated Rb) at final time
        EF_df.loc[i] = EF_row
        RP_df.loc[i] = RP_row
        if (i + 1) % 10 == 0 or i == len(params_df) - 1:
            print(f'  {label}: {i+1}/{len(params_df)} done')

    EF_df['SAMPLE_ID']  = params_df['SAMPLE_ID'].values
    EF_df['PATIENT_ID'] = params_df['PATIENT_ID'].values
    RP_df['SAMPLE_ID']  = params_df['SAMPLE_ID'].values
    RP_df['PATIENT_ID'] = params_df['PATIENT_ID'].values

    return EF_df, RP_df

print('run_ode_for_cohort() defined.')

In [ ]:
# Run ODE for TCGA-GBM patients

patient_params = pd.read_csv('GBM_patient_RbE2Fgenes.csv')
print(f'TCGA-GBM patients: {len(patient_params)}')
patient_params.head()

In [ ]:
print('Running Rb-E2F ODE for TCGA-GBM patients …')
EF_patients, RP_patients = run_ode_for_cohort(patient_params, label='patients')

EF_patients.to_csv('GBM_RbE2F_EF_patients.csv', index=False)
RP_patients.to_csv('GBM_RbE2F_RP_patients.csv', index=False)
print('Saved: GBM_RbE2F_EF_patients.csv and GBM_RbE2F_RP_patients.csv')

# Quick visualisation: E2F dose-response curves for a subset of patients
s_cols = ['S_' + '{:.3f}'.format(s) for s in Svec]
plt.figure(figsize=(8, 5))
for i in range(min(20, len(EF_patients))):
    plt.plot(Svec, EF_patients[s_cols].iloc[i].astype(float), alpha=0.4, color='steelblue')
plt.xscale('log')
plt.xlabel('S (serum/growth stimulus, log scale)')
plt.ylabel('E2F (active, steady-state)')
plt.title('E2F dose-response — TCGA-GBM patients (sample of 20)')
plt.tight_layout()
plt.show()